# LangChain 单 Agent Demo

使用 **DeepSeek V4 Flash**（`deepseek-v4-flash`）+ 本地 Mock 搜索/天气工具，演示查询**上海天气**。

本 notebook 适配 **LangChain 1.x**（`create_agent`），不再依赖已移除的 `langchain.hub` / `create_react_agent` / `AgentExecutor`。

`.env` 只需要 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_BASE_URL`）；工具数据不联网。

## 逻辑总览（彩色 Mermaid）

下图展示 Agent 的执行闭环：LLM 决策 → 需要时调用工具 → 把工具结果写回消息 → 再决策，直到给出最终回答。

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "14px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#64748b"
  }
}}%%
flowchart TD
    U([用户问题]) --> AG[create_agent 图]
    AG --> LLM["deepseek-v4-flash\n决定是否调工具"]

    LLM --> DEC{需要工具?}
    DEC -->|否| FA[最终回答]
    DEC -->|是| ACT[选择 Tool Call]

    ACT --> T1[mock_search]
    ACT --> T2[get_weather_data]

    T1 --> OBS[Tool Message]
    T2 --> OBS
    OBS --> MSG[写回 messages]
    MSG --> LLM

    FA --> OUT([返回 messages])

    classDef user fill:#14b8a6,stroke:#0f766e,stroke-width:2px,color:#042f2e
    classDef exec fill:#38bdf8,stroke:#0284c7,stroke-width:2px,color:#0c4a6e
    classDef model fill:#a78bfa,stroke:#7c3aed,stroke-width:2px,color:#2e1065
    classDef decide fill:#fbbf24,stroke:#d97706,stroke-width:2px,color:#422006
    classDef action fill:#fb7185,stroke:#e11d48,stroke-width:2px,color:#4c0519
    classDef tool fill:#34d399,stroke:#059669,stroke-width:2px,color:#064e3b
    classDef obs fill:#f472b6,stroke:#db2777,stroke-width:2px,color:#500724
    classDef mem fill:#94a3b8,stroke:#475569,stroke-width:2px,color:#0f172a
    classDef answer fill:#4ade80,stroke:#16a34a,stroke-width:2px,color:#14532d

    class U,OUT user
    class AG exec
    class LLM model
    class DEC decide
    class ACT action
    class T1,T2 tool
    class OBS obs
    class MSG mem
    class FA answer
```

**本例路径：** 用户询问「上海天气」→ Agent 调用 `get_weather_data("Shanghai")` → 观测温度/天气/湿度 → 汇总最终回答。

## 1. 导入依赖

引入环境变量加载、DeepSeek 兼容的 ChatOpenAI，以及用于定义本地 Mock 工具的 `@tool` 装饰器。

> LangChain 1.x 中已无 `from langchain import hub`，本 notebook 改为本地 `system_prompt`，无需再拉 Hub 模板。

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool

## 2. 导入 Agent 工厂

LangChain 1.x 使用 `create_agent`：传入 model + tools，返回可 `invoke` 的 Agent 图（内部自动做工具循环）。

In [ ]:
from langchain.agents import create_agent

## 3. 加载环境变量

读取共享 `.env` 中的 DeepSeek 配置；搜索和天气工具使用本地 Mock，不需要额外 Key。

In [ ]:
# ==========================================
# LOAD ENV VARIABLES
# 统一读取最上层：人工智能面试题/.env（从 cwd 向上查找）
# ==========================================
from dotenv import find_dotenv

# os.environ["SSL_CERT_FILE"] = certifi.where()
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path, override=True)
print("loaded .env from:", env_path or "(not found)")

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

## 4. 创建 Mock 搜索工具

返回确定性的本地演示结果，供 Agent 练习工具选择和调用。

In [ ]:
@tool
def search_tool(query: str) -> str:
    """Return local mock search results for a demo."""
    return (
        f"Mock search results for: {query}\n"
        "1. Shanghai demo weather overview: mild and partly cloudy.\n"
        "2. Travel note: carry a light jacket and check real data before going out.\n"
        "Data source: local demonstration fixture (not real-time)"
    )

## 5. 定义天气工具

用 `@tool` 封装本地 Mock 天气数据：传入城市名，返回温度、天气描述和湿度。

In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """
    Return deterministic mock weather for a city.
    """
    presets = {
        "shanghai": (24, "Partly cloudy", 68),
        "上海": (24, "Partly cloudy", 68),
        "beijing": (27, "Sunny", 42),
        "北京": (27, "Sunny", 42),
    }
    temperature, description, humidity = presets.get(
        city.strip().lower(), (22, "Partly cloudy", 60)
    )
    return (
        f"Mock weather for {city}:\n"
        f"Temperature: {temperature} C\n"
        f"Weather: {description}\n"
        f"Humidity: {humidity}%\n"
        "Data source: local demonstration fixture (not real-time)"
    )

## 6. 单独测试搜索工具

先不走 Agent，直接调用本地 Mock，确认工具能正常返回上海相关结果。

In [ ]:
result = search_tool.invoke("Shanghai current weather")
result

## 7. 初始化 LLM

通过 OpenAI 兼容接口连接 DeepSeek V4 Flash，温度设为 0，便于 Agent 稳定选择工具。

In [ ]:
# ==========================================
# LLM — DeepSeek V4 Flash（OpenAI 兼容接口）
# ==========================================

llm = ChatOpenAI(
    model=os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
    temperature=0,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

## 8. 单独测试 LLM

向模型发一条简单请求，确认 API Key 与 base_url 配置正确。

In [ ]:
response = llm.invoke("用一句话介绍上海今天可能的天气特点")
response

## 9. 组装工具列表

把搜索工具和天气工具放进同一个 list，后续一并交给 Agent。

In [ ]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

## 10. 创建 Agent

用 `create_agent` 组合 LLM、工具和系统提示词。返回的是可直接 `invoke` 的图，无需再包一层 AgentExecutor。

In [ ]:
# ==========================================
# CREATE AGENT（LangChain 1.x）
# ==========================================

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. "
        "Use tools when needed to answer questions about weather and current information. "
        "Prefer get_weather_data for weather queries."
    ),
)

## 11. 运行 Agent：查询上海天气

向 Agent 提问「上海当前天气」。输入格式是 `messages` 列表。预期路径：调用 `get_weather_data`（城市为 Shanghai），再输出最终回答。

In [ ]:
# ==========================================
# RUN — 查询上海天气
# ==========================================

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "What is the current weather in Shanghai? "
                "Please report temperature, weather description, and humidity."
            ),
        }
    ]
})

## 12. 打印最终答案

从返回的 `messages` 中取最后一条 AI 消息内容，即为 Agent 汇总后的上海天气说明。

In [ ]:
final_message = result["messages"][-1]
print(final_message.content)

In [ ]:
from pprint import pprint

# 逐条打印消息（最易读）；如需看完整 dict 结构，取消下一行注释
for msg in result["messages"]:
    msg.pretty_print()

# pprint(result, width=100, depth=4)

## 13. 本次运行实际 Workflow（根据日志）

这次运行走的是典型 **ReAct**（Reason + Act）循环：

> **Thought → Action → Observation →（不够则回到 Thought）…**  
> 直到 Observation 足够，才 **出环** 给出 Final Answer。

本例在同一条环上转了 **3 圈**（天气失败 → 搜索不够实时 → refine 成功）后才出环。

### ReAct 循环图（本例转 3 圈）

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "14px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#64748b"
  }
}}%%
flowchart TD
    Q(["Question<br/>上海当前天气？"]) --> TH

    TH["Thought<br/>LLM 推理 / 决定下一步"]
    TH --> DEC{观测够了?}
    DEC -->|否 · 继续环<br/>↻1 ↻2 ↻3| ACT
    DEC -->|是 · 出环| FA

    ACT["Action<br/>发起 Tool Call"]
    ACT --> OBS["Observation<br/>Tool Message 写回"]
    OBS -->|"回到 Thought"| TH

    FA(["Final Answer<br/>31°C · Mostly clear · 湿度 75%"])

    classDef thought fill:#a78bfa,stroke:#7c3aed,stroke-width:2px,color:#2e1065
    classDef action fill:#38bdf8,stroke:#0284c7,stroke-width:2px,color:#0c4a6e
    classDef obs fill:#f472b6,stroke:#db2777,stroke-width:2px,color:#500724
    classDef answer fill:#4ade80,stroke:#16a34a,stroke-width:2px,color:#14532d
    classDef ask fill:#14b8a6,stroke:#0f766e,stroke-width:2px,color:#042f2e
    classDef decide fill:#fbbf24,stroke:#d97706,stroke-width:2px,color:#422006

    class Q ask
    class TH thought
    class DEC decide
    class ACT action
    class OBS obs
    class FA answer
```

### 环上每一圈的载荷（同一循环，不同内容）

| 圈 | Thought | Action | Observation | 决策 |
|---|---|---|---|---|
| ↻1 | 优先用专用天气工具 | `get_weather_data("Shanghai")` | ❌ fetch 失败 | 不够 → 继续环 |
| ↻2 | 天气失败，改搜索 | `tavily_search(current weather…)` | ⚠️ 偏八月预报 | 不够 → 继续环 |
| ↻3 | refine 成 today/now | `tavily_search(today now…)` | ✅ 88°F / 75% | **够了 → 出环** |

**对照日志：** `AIMessage(tool_calls=...)` = Action，`ToolMessage` = Observation，中间自然语言（如 “Let me try looking it up another way”）= Thought。